<a href="https://colab.research.google.com/github/laboratoriodecodigos/Colab-Python/blob/main/Uso_de_Pyspark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("PySparkDemo")
    .getOrCreate()
)

print("Spark:", spark.version)

Spark: 4.0.3


In [3]:
from pyspark.sql.functions import rand
from pyspark.sql.functions import floor
from pyspark.sql.functions import expr

df = (
    spark.range(10_000_000)

    .withColumn(
        "ciudad",
        expr("""
        CASE floor(rand()*6)

            WHEN 0 THEN 'Aguascalientes'
            WHEN 1 THEN 'CDMX'
            WHEN 2 THEN 'Guadalajara'
            WHEN 3 THEN 'Monterrey'
            WHEN 4 THEN 'Puebla'
            ELSE 'León'

        END
        """)
    )

    .withColumn(
        "edad",
        floor(rand()*48)+18
    )

    .withColumn(
        "salario",
        rand()*40000+5000
    )
)

df.show()

+---+--------------+----+------------------+
| id|        ciudad|edad|           salario|
+---+--------------+----+------------------+
|  0|   Guadalajara|  34|6657.3923297677775|
|  1|   Guadalajara|  53| 5278.435993278494|
|  2|Aguascalientes|  64|25157.654539711686|
|  3|Aguascalientes|  26|39376.578371847725|
|  4|Aguascalientes|  48| 13157.29406077553|
|  5|          León|  23| 40712.26298315034|
|  6|          CDMX|  45|34760.324668932735|
|  7|     Monterrey|  62|21773.195539716693|
|  8|   Guadalajara|  52|37238.836728304246|
|  9|          León|  22|25715.152635789014|
| 10|Aguascalientes|  36|11672.598280135506|
| 11|        Puebla|  37| 21282.63069662269|
| 12|          León|  31| 44032.28371523576|
| 13|          CDMX|  26|39637.253467110226|
| 14|          CDMX|  54|  37313.2214476806|
| 15|     Monterrey|  49| 42232.22694361674|
| 16|   Guadalajara|  43|14727.975689488905|
| 17|          León|  65|27557.064234510955|
| 18|     Monterrey|  60|10521.866832687549|
| 19|     

In [4]:
df.printSchema()

root
 |-- id: long (nullable = false)
 |-- ciudad: string (nullable = false)
 |-- edad: long (nullable = true)
 |-- salario: double (nullable = false)



In [5]:
df.count()

10000000

In [6]:
from pyspark.sql.functions import avg
from pyspark.sql.functions import count

(
    df.groupBy("ciudad")
      .agg(
          count("*").alias("empleados"),
          avg("salario").alias("salario_promedio")
      )
      .show()
)

+--------------+---------+------------------+
|        ciudad|empleados|  salario_promedio|
+--------------+---------+------------------+
|     Monterrey|   964611|25000.480810861263|
|   Guadalajara|  1157399| 25000.67217082475|
|Aguascalientes|  1664239|25002.990295125426|
|        Puebla|   805106| 24971.61963325636|
|          CDMX|  1390920|25008.659216011485|
|          León|  4017725|24987.093599166587|
+--------------+---------+------------------+



In [7]:
df.filter(df.edad >= 60).show()

+---+--------------+----+------------------+
| id|        ciudad|edad|           salario|
+---+--------------+----+------------------+
|  2|Aguascalientes|  64|25157.654539711686|
|  7|     Monterrey|  62|21773.195539716693|
| 17|          León|  65|27557.064234510955|
| 18|     Monterrey|  60|10521.866832687549|
| 23|          CDMX|  60| 9135.671377196872|
| 27|   Guadalajara|  64| 33824.27412357359|
| 29|Aguascalientes|  64|19220.447489092458|
| 36|          CDMX|  65|44086.834598278234|
| 52|        Puebla|  63| 5974.715143388663|
| 60|          León|  60|36686.810266592605|
| 61|          León|  65| 26334.40957637409|
| 71|          CDMX|  65| 12820.46241061917|
| 88|          CDMX|  65|14195.536266315947|
| 97|          León|  62|32416.857811555437|
|100|   Guadalajara|  61|29776.074017078234|
|101|   Guadalajara|  62|19857.558821280945|
|123|Aguascalientes|  60|20267.015025585064|
|125|     Monterrey|  62|16825.970522787677|
|130|          León|  60| 7572.056563306044|
|140|     

In [8]:
df.orderBy(df.salario.desc()).show()

+-------+--------------+----+------------------+
|     id|        ciudad|edad|           salario|
+-------+--------------+----+------------------+
|6956589|     Monterrey|  35| 44999.99875450131|
|6527413|          León|  51| 44999.98998406621|
|1088655|Aguascalientes|  25| 44999.98249120355|
|5219552|          CDMX|  33| 44999.97500677042|
|6395714|     Monterrey|  59| 44999.96726113485|
|3966486|     Monterrey|  51| 44999.96629605077|
|9128876|          CDMX|  31|44999.959870223785|
|1020689|Aguascalientes|  57|44999.959036351334|
| 146657|          León|  32| 44999.95736066914|
|3862637|          León|  55| 44999.95393920864|
|5483127|          León|  34| 44999.93522975164|
|5845981|   Guadalajara|  31| 44999.93466369088|
|9908656|          León|  61| 44999.93465839493|
| 505199|          León|  42|44999.933633443885|
|3976248|          CDMX|  40| 44999.92646171234|
|8465831|          CDMX|  54|44999.926236927546|
|6470030|          León|  54| 44999.92123428429|
|7498197|          L

In [9]:
df.createOrReplaceTempView("empleados")

In [10]:
spark.sql("""

SELECT

ciudad,

COUNT(*) AS empleados,

ROUND(AVG(salario),2) AS salario_promedio,

MAX(salario) AS salario_maximo

FROM empleados

GROUP BY ciudad

ORDER BY salario_promedio DESC

""").show()

+--------------+---------+----------------+-----------------+
|        ciudad|empleados|salario_promedio|   salario_maximo|
+--------------+---------+----------------+-----------------+
|          CDMX|  1390920|        25008.66|44999.97500677042|
|Aguascalientes|  1664239|        25002.99|44999.98249120355|
|   Guadalajara|  1157399|        25000.67|44999.93466369088|
|     Monterrey|   964611|        25000.48|44999.99875450131|
|          León|  4017725|        24987.09|44999.98998406621|
|        Puebla|   805106|        24971.62|44999.91235684016|
+--------------+---------+----------------+-----------------+



In [11]:
df.write.mode("overwrite").parquet("empleados_parquet")

In [15]:
df.head(10)

[Row(id=0, ciudad='Guadalajara', edad=34, salario=6657.3923297677775),
 Row(id=1, ciudad='Guadalajara', edad=53, salario=5278.435993278494),
 Row(id=2, ciudad='Aguascalientes', edad=64, salario=25157.654539711686),
 Row(id=3, ciudad='Aguascalientes', edad=26, salario=39376.578371847725),
 Row(id=4, ciudad='Aguascalientes', edad=48, salario=13157.29406077553),
 Row(id=5, ciudad='León', edad=23, salario=40712.26298315034),
 Row(id=6, ciudad='CDMX', edad=45, salario=34760.324668932735),
 Row(id=7, ciudad='Monterrey', edad=62, salario=21773.195539716693),
 Row(id=8, ciudad='Guadalajara', edad=52, salario=37238.836728304246),
 Row(id=9, ciudad='León', edad=22, salario=25715.152635789014)]

In [16]:
import time
from pyspark.sql.functions import rand, floor

inicio = time.time()

df = (
    spark.range(10_000_000)
    .withColumn("edad", floor(rand()*48)+18)
    .withColumn("salario", rand()*40000+5000)
)

df.count()  # Fuerza la ejecución

fin = time.time()

print(f"Tiempo total: {fin-inicio:.2f} segundos")

Tiempo total: 0.27 segundos
